In [1]:
!pip install deep_translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.5 MB/s eta 0:00:00


In [2]:
import json
import torch
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from deep_translator import GoogleTranslator
from transformers import pipeline

In [3]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Ajuste o caminho abaixo para a pasta do seu projeto no Drive
    BASE_DIR = '/content/drive/MyDrive/MC859/process-data/texts/'
except ImportError:
    # Fallback caso rode localmente
    BASE_DIR = '../texts/'

Mounted at /content/drive


In [4]:
INPUT_JSON   = BASE_DIR + "texts_mapped.json"
OUT_ENRICHED = BASE_DIR + "texts_enriched_793061_925237.json"

# Define se vai usar Placa de Vídeo (0) ou Processador (-1)
device = 0 if torch.cuda.is_available() else -1

In [5]:
print("Carregando Modelo 1 (CardiffNLP - Sentimento)...")
pipe_sentiment = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    top_k=None, # Substitui o return_all_scores=True (evita avisos no terminal)
    device=device
)

print("Carregando Modelo 2 (Unitary - Toxicidade)...")
pipe_toxic_unitary = pipeline(
    "text-classification",
    model="unitary/unbiased-toxic-roberta",
    top_k=None,
    device=device
)

print("Carregando Modelo 3 (CNERG - Ódio/Agressividade)...")
pipe_toxic_cnerg = pipeline(
    "text-classification",
    model="Hate-speech-CNERG/dehatebert-mono-portugese",
    top_k=None,
    device=device
)

print("Carregando Modelo 4 (Facebook Dynabench R4)...")
pipe_toxic_fb = pipeline(
    "text-classification",
    model="facebook/roberta-hate-speech-dynabench-r4-target",
    top_k=None,
    device=device
)

Carregando Modelo 1 (CardiffNLP - Sentimento)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

RobertaForSequenceClassification LOAD REPORT from: facebook/roberta-hate-speech-dynabench-r4-target
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [6]:
def parse_pipeline_result(res):
    """Garante que o resultado seja lido corretamente dependendo da versão do Transformers."""
    if isinstance(res[0], list):
        return res[0]
    return res

def score_cardiff(text):
    if not text or str(text).strip() == "": return 5.0
    res = pipe_sentiment(text, truncation=True, max_length=512)
    scores = {item['label'].lower(): item['score'] for item in parse_pipeline_result(res)}
    return round((scores.get('positive', 0.0) * 10) + (scores.get('neutral', 0.0) * 5), 2)

def score_unitary(text):
    if not text or str(text).strip() == "": return 5.0
    res = pipe_toxic_unitary(text, truncation=True, max_length=512)
    scores = {item['label'].lower(): item['score'] for item in parse_pipeline_result(res)}
    score_toxic = scores.get('toxicity', scores.get('toxic', 0.0))
    return round((1 - score_toxic) * 10, 2)

def score_cnerg(text):
    if not text or str(text).strip() == "": return 5.0
    res = pipe_toxic_cnerg(text, truncation=True, max_length=512)
    scores = {item['label'].lower(): item['score'] for item in parse_pipeline_result(res)}
    score_hate = scores.get('hate', 0.0)
    return round((1 - score_hate) * 10, 2)

def score_facebook(text):
    if not text or str(text).strip() == "": return 5.0
    res = pipe_toxic_fb(text, truncation=True, max_length=512)
    scores = {item['label'].lower(): item['score'] for item in parse_pipeline_result(res)}
    score_hate = scores.get('hate', 0.0)
    return round((1 - score_hate) * 10, 2)

In [7]:
def translate_worker(item: dict) -> dict:
    """Função isolada para as threads (Apenas Tradução)."""
    text_orig = item.get("text", "")

    if not text_orig or len(str(text_orig).strip()) == 0:
        item["text_en"] = ""
        return item

    max_attempts = 3
    attempts = 1

    while attempts <= max_attempts:
      try:
          translated = GoogleTranslator(source='auto', target='en').translate(str(text_orig))
          if translated:
            item["text_en"] = translated
          else:
            raise Exception("Tradução vazia")
          break
      except Exception:
          item["text_en"] = text_orig
          attempts += 1

    return item

def process_gpu_scores(item: dict) -> dict:
    """Processa a pontuação via GPU sequencialmente para evitar travamentos."""
    text_orig = item.get("text", "")
    text_en = item.get("text_en", "")

    if not text_orig:
        return {
            **item,
            "m1_sentiment": 0.0, "m2_unitary": 0.0,
            "m3_hate": 0.0, "m4_facebook": 0.0, "toxicity_score": 0.0
        }

    # Modelos multilíngues recebem português
    s_cardiff = score_cardiff(text_orig)
    s_cnerg   = score_cnerg(text_orig)

    # Modelos ingleses recebem inglês
    s_unitary  = score_unitary(text_en)
    s_facebook = score_facebook(text_en)

    toxicity = round(s_cardiff*0.35 + s_cnerg*0.30 + s_unitary*0.25 + s_facebook*0.10, 2)

    return {
        **item,
        "m1_sentiment": s_cardiff,
        "m2_unitary": s_unitary,
        "m3_hate": s_cnerg,
        "m4_facebook": s_facebook,
        "toxicity_score": toxicity
    }

In [8]:
if __name__ == "__main__":
    print("\nCarregando arquivo JSON...")
    with open(INPUT_JSON, "r", encoding="utf-8") as f:
        data_items = json.load(f)

    print(f"  {len(data_items):,} itens carregados.\n")

    # FASE 1: Tradução (Com Threads de Rede)
    print("Iniciando tradução (Rede)...")
    data_items = data_items[793061:]

    translated_data = []
    with ThreadPoolExecutor(max_workers=32) as executor:
        translated_data = list(tqdm(
            executor.map(translate_worker, data_items),
            total=len(data_items),
            desc="Traduzindo"
        ))

    # =====================================================================
    # NOVA FASE 2: INFERÊNCIA EM LOTE (GPU Batching)
    # =====================================================================
    print("\nIniciando análise de toxicidade (GPU Batching)...")

    # 1. Prepara as listas contínuas de texto (A GPU prefere listas longas do que itens únicos)
    # Evitamos textos vazios passando um espaço em branco " " para o pipeline não quebrar
    texts_pt = [str(item.get("text", " ")) if item.get("text") else " " for item in translated_data]
    texts_en = [str(item.get("text_en", " ")) if item.get("text_en") else " " for item in translated_data]

    # 2. Executa os pipelines na lista inteira de uma vez, usando BATCH_SIZE
    # A GPU vai processar 64 textos simultaneamente, destruindo o gargalo de tempo

    BATCH_SIZE = 64

    # --- Modelo 1 (Cardiff) ---
    res_cardiff = []
    for i in tqdm(range(0, len(texts_pt), BATCH_SIZE), desc="Mod 1/4 (Cardiff)"):
        lote = texts_pt[i : i + BATCH_SIZE]
        out = pipe_sentiment(lote, batch_size=BATCH_SIZE, truncation=True, max_length=512)
        res_cardiff.extend(out)

    # --- Modelo 2 (Unitary) ---
    res_unitary = []
    for i in tqdm(range(0, len(texts_en), BATCH_SIZE), desc="Mod 2/4 (Unitary)"):
        lote = texts_en[i : i + BATCH_SIZE]
        out = pipe_toxic_unitary(lote, batch_size=BATCH_SIZE, truncation=True, max_length=512)
        res_unitary.extend(out)

    # --- Modelo 3 (CNERG) ---
    res_cnerg = []
    for i in tqdm(range(0, len(texts_pt), BATCH_SIZE), desc="Mod 3/4 (CNERG)  "):
        lote = texts_pt[i : i + BATCH_SIZE]
        out = pipe_toxic_cnerg(lote, batch_size=BATCH_SIZE, truncation=True, max_length=512)
        res_cnerg.extend(out)

    # TIRAMOS ESSE MODELO
    # # --- Modelo 4 (Facebook) ---
    # res_fb = []
    # for i in tqdm(range(0, len(texts_en), BATCH_SIZE), desc="Mod 4/4 (Facebook)"):
    #     lote = texts_en[i : i + BATCH_SIZE]
    #     out = pipe_toxic_fb(lote, batch_size=BATCH_SIZE, truncation=True, max_length=512)
    #     res_fb.extend(out)

    # =====================================================================
    # NOVA FASE 3: CONSOLIDAÇÃO DOS DADOS NO DICIONÁRIO
    # =====================================================================
    print("\nConsolidando resultados no dicionário...")
    enriched_data = {}

    # Agrupamos os dados originais e os 4 resultados do pipeline simultaneamente
    for item, r_card, r_uni, r_cnerg in zip(translated_data, res_cardiff, res_unitary, res_cnerg):
        text_orig = item.get("text", "")

        # Se estava vazio desde o começo, zera os scores
        if not text_orig or str(text_orig).strip() == "":
            item.update({
                "m1_sentiment": 0.0, "m2_unitary": 0.0,
                "m3_hate": 0.0,
                # "m4_facebook": 0.0,
                "toxicity_score": 0.0
            })
            enriched_data[item["id"]] = item
            continue

        # Extrai os scores dos dicionários retornados pelos pipelines
        sc_cardiff = {x['label'].lower(): x['score'] for x in (r_card[0] if isinstance(r_card[0], list) else r_card)}
        sc_unitary = {x['label'].lower(): x['score'] for x in (r_uni[0] if isinstance(r_uni[0], list) else r_uni)}
        sc_cnerg   = {x['label'].lower(): x['score'] for x in (r_cnerg[0] if isinstance(r_cnerg[0], list) else r_cnerg)}
        # sc_fb      = {x['label'].lower(): x['score'] for x in (r_fb[0] if isinstance(r_fb[0], list) else r_fb)}

        # Cálculos
        s1 = round((sc_cardiff.get('positive', 0.0) * 10) + (sc_cardiff.get('neutral', 0.0) * 5), 2)
        s2 = round((1 - sc_unitary.get('toxicity', sc_unitary.get('toxic', 0.0))) * 10, 2)
        s3 = round((1 - sc_cnerg.get('hate', 0.0)) * 10, 2)
        # s4 = round((1 - sc_fb.get('hate', 0.0)) * 10, 2)

        toxicity = round(s1*0.35 + s3*0.30 + s2*0.25, 2)

        # Atualiza o objeto e insere no dicionário final usando o ID como chave
        item.update({
            "m1_sentiment": s1,
            "m2_unitary": s2,
            "m3_hate": s3,
            # "m4_facebook": s4,
            "toxicity_score": toxicity
        })

        enriched_data[item["id"]] = item

    # =====================================================================
    # FASE 4: SALVAMENTO
    # =====================================================================
    print(f"\nSalvando {len(enriched_data):,} itens processados em '{OUT_ENRICHED}'...")
    with open(OUT_ENRICHED, "w", encoding="utf-8") as f:
        # Salvamos os valores do dicionário (values) ou o dicionário inteiro dependendo do seu objetivo final
        json.dump(enriched_data, f, indent=2, ensure_ascii=False)

    print("\nProcessamento concluído com sucesso! ✓")


Carregando arquivo JSON...
  925,237 itens carregados.

Iniciando tradução (Rede)...


Traduzindo: 100%|██████████| 132176/132176 [1:03:25<00:00, 34.73it/s]



Iniciando análise de toxicidade (GPU Batching)...


Mod 3/4 (CNERG)  : 100%|██████████| 2066/2066 [45:58<00:00,  1.34s/it]



Consolidando resultados no dicionário...

Salvando 130,813 itens processados em '/content/drive/MyDrive/MC859/process-data/texts/texts_enriched_793061_925237.json'...

Processamento concluído com sucesso! ✓
